In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
dataset_path = os.path.join(BASE_DIR, "hand_gesture_dataset_processed")

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir = os.path.join(dataset_path, "val")
test_dir = os.path.join(dataset_path, "test")

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=False
)

Found 490 images belonging to 7 classes.
Found 140 images belonging to 7 classes.
Found 70 images belonging to 7 classes.


In [ ]:
num_classes = len(train_data.class_indices)

In [ ]:
with open(os.path.join(BASE_DIR, "class_indices.json"), "w") as f:
    json.dump(train_data.class_indices, f)

In [ ]:
model = models.Sequential([
    # 1. Input Layer + First Convolution
    # We use 32 filters of size 3x3. Input is 50x50 with 1 channel (gray)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # 2. Second Convolution (extracts more complex features)
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # 3. Third Convolution
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # 4. Flattening (converts 2D 2D features into a 1D vector for the classifier)
    layers.Flatten(),
    
    # 5. Dense Layers (The Classifier)
    layers.Dense(64, activation='relu'),
    
    # 6. Output Layer
    # Use 'softmax' for multi-class classification
    layers.Dense(num_classes, activation='softmax')
])

c:\Users\louis\OneDrive\Documents\Telerobotics\computer_vision_speed_control\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 9, 9, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5184)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       331,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,039 (1.48 MB)

 Trainable params: 388,039 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
checkpoint_path = os.path.join(BASE_DIR, "hand_gesture_model_best.keras")
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
callbacks = [checkpoint, earlystop]

In [ ]:
print("\nTraining the model...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


Training the model...
Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.2766 - loss: 1.8859
Epoch 1: val_loss improved from None to 1.33949, saving model to c:\Users\louis\OneDrive\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 1: finished saving model to c:\Users\louis\OneDrive\Documents\Telerobotics\hand_gesture_model_best.keras
16/16 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - accuracy: 0.4184 - loss: 1.7672 - val_accuracy: 0.6429 - val_loss: 1.3395
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6875 - loss: 1.1268
Epoch 2: val_loss improved from 1.33949 to 0.52126, saving model to c:\Users\louis\OneDrive\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 2: finished saving model to c:\Users\louis\OneDrive\Documents\Telerobotics\hand_gesture_model_best.keras
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.7245 - loss: 0.9134 - val_accuracy: 0.8000 - val_loss: 0.5213
Epoch 3/20
15/16 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 

In [ ]:
print("\nEvaluating on test data...")
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Evaluating on test data...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9429 - loss: 0.0992 
Test Loss: 0.0992
Test Accuracy: 0.9429


In [ ]:
y_true = test_data.classes
preds = model.predict(test_data, verbose=1)
y_pred = preds.argmax(axis=1)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


In [ ]:
cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred, target_names=[k for k, v in sorted(train_data.class_indices.items(), key=lambda x: x[1])])
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", cr)
with open(os.path.join(BASE_DIR, "classification_report.txt"), "w") as f:
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\nClassification Report:\n")
    f.write(cr)


Confusion Matrix:
 [[10  0  0  0  0  0  0]
 [ 0 10  0  0  0  0  0]
 [ 0  3  7  0  0  0  0]
 [ 0  0  0 10  0  0  0]
 [ 0  0  0  0 10  0  0]
 [ 0  0  0  0  0 10  0]
 [ 1  0  0  0  0  0  9]]

Classification Report:
                       precision    recall  f1-score   support

    left_hand_images       0.91      1.00      0.95        10
level_1_speed_images       0.77      1.00      0.87        10
level_2_speed_images       1.00      0.70      0.82        10
level_3_speed_images       1.00      1.00      1.00        10
level_4_speed_images       1.00      1.00      1.00        10
   right_hand_images       1.00      1.00      1.00        10
         stop_images       1.00      0.90      0.95        10

            accuracy                           0.94        70
           macro avg       0.95      0.94      0.94        70
        weighted avg       0.95      0.94      0.94        70



In [ ]:
plt.figure()
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.legend()
plt.title('Loss')
plt.savefig(os.path.join(BASE_DIR, 'loss_curve.png'))
plt.close()

In [ ]:
plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.savefig(os.path.join(BASE_DIR, 'accuracy_curve.png'))
plt.close()

In [ ]:
model_save_path = os.path.join(BASE_DIR, "hand_gesture_model.keras")
model.save(model_save_path)
print(f"\nModel saved to {model_save_path}")


Model saved to c:\Users\louis\OneDrive\Documents\Telerobotics\hand_gesture_model.keras
